# Jetflix — 01 Data Exploration

**Purpose:** Inspect every raw dataset before any cleaning, feature engineering, or modelling.

**Data window:** 2025-10-01 to 2026-09-30  
**Extract date:** 2026-10-01  
**Sample:** 50,000 accounts that held an active subscription at any point in the window  
**Files:** `data/raw/*.csv.gz` (UTF-8, gzipped CSV, header row)  

**Retention notes:**
- `playback_events`, `playback_quality`, `clickstream` — last **90 days** only (2026-07-03 to 2026-09-30).
- Account-level tables — full history from account creation (as far back as 2023-10-01).
- Activity tables — full analysis window (2025-10-01 to 2026-09-30).

**Conventions in raw data:**
- Empty field = NULL.
- `-1` = system did not capture or could not measure (not an error).
- Timestamps are ISO-8601 UTC; dates are `YYYY-MM-DD`.

**Rules:** Do not modify raw files. Do not create churn labels or model features.


In [ ]:
# Shared imports and helpers
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RAW = "../data/raw"
RANDOM_SEED = 42
SAMPLE_N = 5
TOP_N = 10


def inspect(name, date_cols=None, nrows=None):
    """Load a gzipped CSV and print a full inspection report.

    Parameters
    ----------
    name       : table name without .csv.gz
    date_cols  : list of column names to parse as datetime
    nrows      : limit rows loaded (None = all)
    """
    path = f"{RAW}/{name}.csv.gz"
    sep = "=" * 70
    print(f"\n{sep}")
    print(f"  TABLE: {name}   [{path}]")
    print(sep)

    df = pd.read_csv(path, low_memory=False, nrows=nrows)
    rows, cols = df.shape
    label = f"{rows:,} rows x {cols} columns"
    if nrows:
        label += f"  (WARNING: only first {nrows:,} rows loaded)"
    print(f"\nShape: {label}")

    mem_mb = df.memory_usage(deep=True).sum() / 1_048_576
    print(f"Approximate memory: {mem_mb:.2f} MB")

    print("\nColumns and dtypes:")
    display(df.dtypes.rename("dtype").to_frame().T)

    print("\nFirst 5 rows:")
    display(df.head())

    n_sample = min(SAMPLE_N, rows)
    print(f"\nRandom sample ({n_sample} rows):")
    display(df.sample(n_sample, random_state=RANDOM_SEED))

    # -- Missing values --------------------------------------------------
    print("\nMissing values:")
    null_count = df.isnull().sum()
    null_pct = (null_count / rows * 100).round(2)
    missing_df = pd.DataFrame({"missing_count": null_count, "missing_%": null_pct})
    missing_df = missing_df[missing_df["missing_count"] > 0].sort_values(
        "missing_count", ascending=False
    )
    if missing_df.empty:
        print("  No missing values.")
    else:
        display(missing_df)

    # -- Sentinel -1 -----------------------------------------------------
    num_cols_list = df.select_dtypes(include="number").columns
    neg1_cols = [
        (c, int((df[c] == -1).sum()))
        for c in num_cols_list
        if (df[c] == -1).any()
    ]
    if neg1_cols:
        print("\nColumns with sentinel -1 (means 'not captured', NOT an error):")
        for c, n in neg1_cols:
            pct = n / rows * 100
            print(f"  {c}: {n:,} rows ({pct:.1f}%)")

    # -- Duplicates ------------------------------------------------------
    dup = int(df.duplicated().sum())
    pct_dup = dup / rows * 100
    print(f"\nFully duplicated rows: {dup:,} ({pct_dup:.2f}%)")

    # -- Unique counts ---------------------------------------------------
    print("\nUnique values per column:")
    display(df.nunique().rename("unique_count").to_frame().T)

    # -- Numerical summary -----------------------------------------------
    num_df = df.select_dtypes(include="number")
    if not num_df.empty:
        print("\nNumerical summary:")
        display(num_df.describe().round(4))

    # -- Categorical top values ------------------------------------------
    cat_df = df.select_dtypes(include="object")
    if not cat_df.empty:
        print(f"\nTop {TOP_N} values in categorical / text columns:")
        for c in cat_df.columns:
            vc = df[c].value_counts(dropna=False).head(TOP_N)
            print(f"  -- {c} --")
            for val, cnt in vc.items():
                pct = cnt / rows * 100
                print(f"     {str(val)!r:40s} {cnt:>8,} ({pct:.1f}%)")

    # -- Date ranges -----------------------------------------------------
    dt_auto = [
        c for c in df.columns
        if any(c.endswith(s) for s in (
            "_ts", "_date", "added_ts", "removed_ts", "closed_ts", "opened_ts"
        ))
    ]
    dt_cols_all = list(dict.fromkeys((date_cols or []) + dt_auto))
    if dt_cols_all:
        print("\nDate / timestamp columns and ranges:")
        for c in dt_cols_all:
            if c not in df.columns:
                continue
            try:
                parsed = pd.to_datetime(df[c], errors="coerce", utc=True)
                n_bad = int(max(parsed.isnull().sum() - df[c].isnull().sum(), 0))
                non_null = parsed.dropna()
                if non_null.empty:
                    print(f"  {c}: all NULL or unparseable")
                else:
                    lo, hi = non_null.min(), non_null.max()
                    print(f"  {c}: {lo} to {hi}  (unparseable: {n_bad:,})")
            except Exception as exc:
                print(f"  {c}: could not parse -- {exc}")

    # -- ID columns ------------------------------------------------------
    id_cols = [c for c in df.columns if c.endswith("_id") or c == "invoice_id"]
    if id_cols:
        print("\nID columns -- uniqueness check:")
        for c in id_cols:
            nn = int(df[c].notna().sum())
            uq = int(df[c].nunique())
            rep = nn - uq
            print(
                f"  {c:35s} non-null: {nn:>8,}  unique: {uq:>8,}  repeated: {rep:>6,}"
            )

    print(f"\n{'-' * 70}\n")
    return df


import sys
print("Helper loaded.")
print(f"Python {sys.version.split()[0]}  |  Pandas {pd.__version__}  |  NumPy {np.__version__}")
print(f"Raw data folder: {RAW}")

---
## 1. accounts

Account owner record as of extraction. One row per account. **Snapshot table.**


In [ ]:
df_accounts = inspect(
    'accounts',
    date_cols=['account_created_ts'],
)


---
## 2. profiles

One row per profile (purged profiles removed). **Snapshot table.**


In [ ]:
df_profiles = inspect(
    'profiles',
    date_cols=['created_ts', 'deleted_ts'],
)


---
## 3. devices

One row per registered device. **Snapshot table.**

> Note: `device_category` is not normalised — same device class may appear as `TV`, `Smart TV`, or `tv`.


In [ ]:
df_devices = inspect(
    'devices',
    date_cols=['first_seen_ts', 'last_seen_ts', 'deactivated_ts'],
)


---
## 4. payment_methods

Every payment method ever added, including removed ones. **Full history.**


In [ ]:
df_payment_methods = inspect(
    'payment_methods',
    date_cols=['added_ts', 'last_updated_ts', 'removed_ts'],
)


---
## 5. subscription_history

One row per subscription period. Members who leave and return get a new row.


In [ ]:
df_subscription_history = inspect(
    'subscription_history',
    date_cols=['opened_ts', 'cancel_requested_ts', 'closed_ts'],
)


---
## 6. billing_history

Every charge attempt, retry, and \$0 verification. **Full history. Large (~677 k rows).**


In [ ]:
df_billing_history = inspect(
    'billing_history',
    date_cols=['transaction_ts', 'billing_period_start', 'billing_period_end', 'next_billing_date'],
)


---
## 7. membership_events

Account change log from the membership service. **Full history.**


In [ ]:
df_membership_events = inspect(
    'membership_events',
    date_cols=['event_ts'],
)


---
## 8. consent_events

Every consent change, including the initial value set at signup. **Full history.**

> Composite primary key: `(account_id, consent_type, changed_ts)`.


In [ ]:
df_consent_events = inspect(
    'consent_events',
    date_cols=['changed_ts'],
)


---
## 9. viewing_activity

One row per playback. Full analysis window. **Very large (~3.3 M rows).**


In [ ]:
df_viewing_activity = inspect(
    'viewing_activity',
    date_cols=['start_ts'],
)


---
## 10. playback_events

Play-trace events within a view (start, pause, reposition, stop). **Last 90 days only.**

> ~0.25% rows are expected client-re-send duplicates.


In [ ]:
df_playback_events = inspect(
    'playback_events',
    date_cols=['client_ts', 'server_received_ts'],
)


---
## 11. playback_quality

QoE telemetry per playback attempt, including failed attempts. **Last 90 days only.**

> Sentinel `-1`: `startup_time_ms` (failed attempt) and `rebuffer_duration_ms` (old client). These are **not** errors.


In [ ]:
df_playback_quality = inspect(
    'playback_quality',
    date_cols=['client_ts'],
)


---
## 12. search_history

One row per search action. Full analysis window.


In [ ]:
df_search_history = inspect(
    'search_history',
    date_cols=['search_ts'],
)


---
## 13. clickstream

App and web UI events. **Last 90 days only. Large (~2.2 M rows).**

> ~0.3% rows are expected duplicates. App-store cancels produce no Jetflix clickstream.


In [ ]:
df_clickstream = inspect(
    'clickstream',
    date_cols=['client_ts', 'server_received_ts'],
)


---
## 14. ratings

Thumbs ratings on titles. Full analysis window.


In [ ]:
df_ratings = inspect(
    'ratings',
    date_cols=['event_ts'],
)


---
## 15. my_list

Watchlist (My List) adds and removes. Full analysis window.


In [ ]:
df_my_list = inspect(
    'my_list',
    date_cols=['added_ts', 'removed_ts'],
)


---
## 16. login_events

Sign-ins, failures, logouts, password resets. Full analysis window.


In [ ]:
df_login_events = inspect(
    'login_events',
    date_cols=['event_ts'],
)


---
## 17. support_contacts

Customer-service contacts. Full analysis window.


In [ ]:
df_support_contacts = inspect(
    'support_contacts',
    date_cols=['contact_ts'],
)


---
## 18. communications_log

All outbound messages (email, SMS, push). Full analysis window. **Large (~893 k rows).**


In [ ]:
df_communications_log = inspect(
    'communications_log',
    date_cols=['sent_ts', 'delivered_ts', 'opened_ts', 'clicked_ts', 'unsubscribed_ts'],
)


---
## 19. catalog

Reference table of all titles. 3,345 rows.


In [ ]:
df_catalog = inspect(
    'catalog',
    date_cols=['available_from', 'available_until'],
)


---
## 20. episodes

Reference table of episodes for series titles. 25,240 rows.


In [ ]:
df_episodes = inspect(
    'episodes',
    date_cols=['release_date'],
)
